# Experiment B: Custom YOLOv8n Fine-Tuning on Gate Dataset
### AI-Based Smart Gate Monitoring System
**Author:** Siddharth Dhanush R (Team Lead, AI Training, Backend & Integration)  
**Objective:** Fine-tune pretrained `yolov8n.pt` on the campus gate surveillance dataset for 50 epochs using Google Colab T4 GPU acceleration, with negative background sampling to eliminate false positives.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/siddharthdhanushr-hub/AI-Based-Smart-Gate-Monitoring-System-Using-Real-Time-Object-Detection-and-Tracking-/blob/main/ai/notebooks/03_custom_training.ipynb)

---
### Instructions to Run in 3 Clicks:
1. Click the **Open in Colab** badge above.
2. Navigate to **Runtime -> Change runtime type -> T4 GPU** and click Save.
3. Click **Runtime -> Run all**.
4. When training finishes, `best.pt` will automatically download to your browser! Drop it into `ai/models/gate_yolov8n.pt`.

## 1. Verify Google Colab GPU & Environment

In [ ]:
!pip install -q ultralytics pyyaml

import os
import sys
import shutil
import torch
from ultralytics import YOLO

print(f"[INFO] PyTorch Version: {torch.__version__}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"[SUCCESS] Active GPU: {gpu_name} ({vram_gb:.2f} GB VRAM)")
    batch_size = 16 if vram_gb >= 11.0 else 8
    device_target = 0
else:
    print("[WARNING] CUDA GPU not detected! Switch via: Runtime -> Change runtime type -> T4 GPU.")
    batch_size = 8
    device_target = 'cpu'

print(f"[INFO] Selected Batch Size: {batch_size}, Device: {device_target}")

## 2. Synchronize Project Repository

In [ ]:
# Ensure repository files and scripts are accessible in Colab
if not os.path.exists('ai/dataset/sample_gate.mp4') and not os.path.exists('../ai/dataset/sample_gate.mp4'):
    print("[INFO] Cloning project repository into Colab environment...")
    !git clone https://github.com/siddharthdhanushr-hub/AI-Based-Smart-Gate-Monitoring-System-Using-Real-Time-Object-Detection-and-Tracking-.git repo
    %cd repo
    print(f"[INFO] Active Working Directory: {os.getcwd()}")
else:
    print(f"[INFO] Running inside repository tree: {os.getcwd()}")

# Ensure Python path includes root for module imports
sys.path.insert(0, os.getcwd())

## 3. Dataset Setup & Negative Sample Generation
Supports dual dataset workflows:
- **Option A (Google Drive)**: Loads user's custom annotated gate dataset from `/content/drive/MyDrive/GateDataset/data.yaml` if available.
- **Option B (Automated Surveillance Pipeline)**: Extracts frames from `sample_gate.mp4` and embeds **negative background samples** (empty labels) to explicitly train the network never to flag ceiling fans, indoor fixtures, or plain backgrounds as vehicles!

In [ ]:
DATA_YAML_PATH = None

# Check Option A: Google Drive custom dataset
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    drive_yaml = '/content/drive/MyDrive/GateDataset/data.yaml'
    if os.path.exists(drive_yaml):
        DATA_YAML_PATH = drive_yaml
        print(f"[INFO] Successfully loaded custom dataset from Google Drive: {DATA_YAML_PATH}")
except Exception:
    pass

# Option B: Build automated gate surveillance dataset
if not DATA_YAML_PATH or not os.path.exists(DATA_YAML_PATH):
    print("[INFO] Google Drive dataset not found. Generating self-contained gate surveillance dataset with negative background suppression...")
    from ai.src.dataset_utils import prepare_colab_surveillance_dataset
    
    video_source = 'ai/dataset/sample_gate.mp4'
    colab_dataset_root = os.path.abspath('ai/dataset/colab_run_dataset')
    
    DATA_YAML_PATH = prepare_colab_surveillance_dataset(
        video_path=video_source,
        output_root=colab_dataset_root,
        stride=15,
        max_frames=120,
        include_background_negatives=True
    )
    print(f"[SUCCESS] Prepared gate dataset at: {DATA_YAML_PATH}")

# Inspect data.yaml
with open(DATA_YAML_PATH, 'r') as f:
    print(f.read())

## 4. Initialize Pretrained YOLOv8n Weights

In [ ]:
# Start transfer learning from official lightweight YOLOv8 nano weights
model = YOLO('yolov8n.pt')
print("[INFO] Pretrained YOLOv8n weights loaded successfully.")

## 5. Execute Fine-Tuning (50 Epochs on GPU)

In [ ]:
print(f"[INFO] Launching fine-tuning on {DATA_YAML_PATH} using device {device_target}...")
results = model.train(
    data=DATA_YAML_PATH,
    epochs=50,
    imgsz=640,
    batch=batch_size,
    patience=10,         # Early stopping if no mAP improvement
    save=True,
    project='gate_monitoring_runs',
    name='yolov8n_custom_gate',
    device=device_target,
    workers=2,
    verbose=True
)
print("[SUCCESS] Fine-tuning run complete!")

## 6. Validate Model on Held-Out Test Set

In [ ]:
best_weights = 'gate_monitoring_runs/yolov8n_custom_gate/weights/best.pt'
if os.path.exists(best_weights):
    print("[INFO] Evaluating fine-tuned model on test split...")
    trained_model = YOLO(best_weights)
    metrics = trained_model.val(data=DATA_YAML_PATH, split='test', device=device_target)
    print(f"mAP@50: {metrics.box.map50:.4f}")
    print(f"mAP@50-95: {metrics.box.map:.4f}")
else:
    print(f"[WARNING] Best weights not found at: {best_weights}")

## 7. Direct Browser Download & Google Drive Export

In [ ]:
best_weights = 'gate_monitoring_runs/yolov8n_custom_gate/weights/best.pt'

if os.path.exists(best_weights):
    # Optional: Backup to Google Drive if connected
    target_drive_path = '/content/drive/MyDrive/GateMonitoringModels/gate_yolov8n_best.pt'
    try:
        os.makedirs(os.path.dirname(target_drive_path), exist_ok=True)
        shutil.copy(best_weights, target_drive_path)
        print(f"[SUCCESS] Saved backup to Google Drive at: {target_drive_path}")
    except Exception:
        pass

    # Direct browser download in Google Colab
    try:
        from google.colab import files
        print("[INFO] Initiating direct browser download for best.pt...")
        files.download(best_weights)
        print("[SUCCESS] Download dialog triggered! Save the file and move to ai/models/gate_yolov8n.pt")
    except Exception as e:
        print(f"[INFO] Please download manually from {best_weights}: {e}")
else:
    print(f"[ERROR] Checkpoint weights not found at: {best_weights}")